- 有数据时，先明确每行每列以及单位
- 检查缺失值
  - 异常值（离谱的，需判断真实极端还是录入错误）
  - 重复值（记录多次）
  - 数据类型错误（数字类型变成字符串有问题）
- 把不同单位，不同方向，不同尺度数据转换为可比较形式
- 通过统计量、图表、相关性分析，为模型选择提供依据
- 评委不仅看结果，还看数据来源、处理依据和是否可复现
- 目标：让数据被模型正确使用，将附件整理为可直接使用的数值矩阵

- X=[x_ij]_n*m
- n个评价对象，m个指标，x_ij表示第i个对象在第j个指标上的观测值

- 对象——指标——数据矩阵——模型输入——结果输出

- 注意：
  - 缺失值一定要处理，指标方向要统一（越大/小越好）
  - 量纲要消除

- 每一行称样本，每一列称指标，值为观测结果

- 常见数据格式：
  - Excel：`.xlsx` → `pd.read_excel(路径)`
  - CSV：`.csv` → `pd.read_csv(路径)`
- 图片文本先提取信息，一般不需要导入

- 读取数据检查顺序：
  - 文件是否存在
  - 编码是否正确（是否乱码，重下）
  - 列名是否完整
  - 数据类型是否合理
  - 是否可保存

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

#要先把数据放vscode里
DATA_PATH=Path('d:\\git项目\\导入数据')
OUTPUT_DIR=Path('d:\\git项目\\输出数据')

print(Path.cwd())#输出当前工作目录
print(DATA_PATH.exists())#判断导入路径是否存在
print(OUTPUT_DIR.exists())#判断输出路径是否存在


In [ ]:
#把附件变成DataFrame
df_raw=pd.read_csv(DATA_PATH/'初始资金为10000的资金变化.csv')
df_raw

In [ ]:
print(df_raw.shape)#输出数据的行列数
df_raw.info()#输出数据的基本信息

- 数据清洗学习
  - 1、把错误不完整不一致和异常记录进行检查和处理
    - 查看字段——检查确实——检查重复——检查类型——识别异常——记录处理
  - 2、清洗原则：说清楚每一步原因，并能在这里复现（用代码）
    - 原始附件——处理代码——输出表格
  - 3、论文中说明：处理对象、处理方法、处理依据，以及处理前后变化

- 缺失值常见来源：
  - 1、数据采集时没记录
  - 2、问卷中受访者未填写
  - 3、设备故障中断
  - 4、多个数据表合并字段不匹配
  - 5、原始数据中使用特殊符号表示缺测
- 影响均值、方差、相关系数、回归模型、聚类模型和综合评价结果
- 缺失值出现在某一类样本中时，随意删除可能造成样本偏差，处理缺失值关键在于保留数据含义并减少偏差

- 缺失率=缺失值个数/样本总数*100%
- 0、某一列缺失率高时可能直接删除列，不删除行
- 1、缺失很少且不影响样本代表性时删除行，但一般连续时间序列不可随便删去，可能会造成时间跨度不一致
- 2、缺失值少时可用均值、中位数、众数、分组中位数填补
  - （差异明显时按区域分组，计算中位数，填补缺失再检查）尽量尊重数据分组特征
- 3、时间序列或有顺序的数据可用插值（线性插值等）
- 4、缺失本身有含义不能机械填补要单独说明

- df.duplicated()检查有无重复录入数据
- df.drop_duplicates()删除重复行

In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import numpy as np
import pandas as pd

IN_DIR = Path(r"d:\git项目\导入数据")
OUT_DIR = Path(r"d:\git项目\输出数据")
NAME = "mcm_final_raw"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_any(folder, stem, order=(".csv", ".xlsx")):
    for ext in order:                       # csv / xlsx 都能读，优先 csv
        p = folder / f"{stem}{ext}"
        if p.exists():
            print("读取：", p)
            return pd.read_excel(p) if ext == ".xlsx" else pd.read_csv(p)
    raise FileNotFoundError(f"{folder} 下找不到 {stem}.csv / .xlsx")

def dedup(d, key, num):
    """规范化后调用：删完全重复 + 主键冲突按确定性规则保留一条"""
    n = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first")
    conf = d.groupby(key)[num].nunique(dropna=False).gt(1).any(axis=1)
    med = d[num].median()
    d = d.assign(_ok=d[num].notna().sum(axis=1),          # 有效字段多者优先
                 _dev=d[num].sub(med).abs().sum(axis=1))  # 偏离典型值小者优先
    d = (d.sort_values(["_ok", "_dev"] + key,
                       ascending=[False, True] + [True] * len(key), kind="mergesort")
          .drop_duplicates(subset=key, keep="first")
          .drop(columns=["_ok", "_dev"]).reset_index(drop=True))
    print(f"完全重复 {n} 行已删 | 主键冲突组 {int(conf.sum())} 个 | 去重后 {d.shape}")
    return d

df = load_any(IN_DIR, NAME)
print("原始：", df.shape)

# ① 规范化：特殊符号 -> NaN，数值列转数值，文本列去空格
df = df.replace(["", "NA", "缺测", "未填", "9999", 9999], np.nan)
num = ["pm25", "temperature", "traffic_index", "survey_score", "household_income"]
df[num] = df[num].apply(pd.to_numeric, errors="coerce")
for c in ["date", "region", "fuel_type"]:
    df[c] = df[c].astype("string").str.strip()

# ② 去重（必须先于填补，否则重复行会拉偏中位数/均值）
key = ["date", "region"]
df = dedup(df, key, num)
assert not df.duplicated().any() and not df.duplicated(subset=key).any()

# ③ 缺失处理：删列 -> 填补 -> 插值 -> 有意义缺失
rate = df.isna().mean() * 100
print("缺失率(%)：\n" + rate.round(2).to_string())
df = df.drop(columns=rate[rate >= 50].index)                        # 情况0 删列不删行
# 情况1 时间序列不删行，保留全部时间点
df["traffic_index"] = df["traffic_index"].fillna(df["traffic_index"].median())
df["fuel_type"] = df["fuel_type"].fillna(df["fuel_type"].mode()[0])
g = df.groupby("region")["pm25"].transform("median")                  # 情况2 分组中位数
df["pm25"] = df["pm25"].fillna(g).fillna(df["pm25"].median())
df = df.sort_values(["region", "date"])
df["temperature"] = df.groupby("region")["temperature"].transform(    # 情况3 插值
    lambda s: s.interpolate(method="linear", limit_direction="both"))
df["survey_score_missing"] = df["survey_score"].isna().astype(int)     # 情况4 保留 NaN

# ④ 复核 + 唯一最终输出
assert df[["pm25", "temperature", "traffic_index", "fuel_type"]].isna().sum().sum() == 0
print("处理后缺失率(%)：\n" + (df.isna().mean() * 100).round(2).to_string())
df.to_csv(OUT_DIR / "final_cleaned.csv", index=False, encoding="utf-8-sig")
df.to_excel(OUT_DIR / "final_cleaned.xlsx", index=False, sheet_name="最终结果")
print("唯一最终版：", OUT_DIR / "final_cleaned.xlsx")


Data preprocessing. All non-standard missing markers (NA, 缺测, 未填, 9999, etc.) were converted to NaN, and text fields were stripped of leading/trailing spaces before de-duplication. We removed 15 exact duplicate rows and resolved 6 key conflicts on (date, region) using a deterministic rule that keeps the record with more valid fields and the smallest deviation from the overall median. The resulting panel contains 360 daily observations (4 regions × 90 consecutive days). Missing rate is defined as the number of missing cells divided by the total sample size, n = 360. The column household_income (88.61% missing) was dropped; columns, not rows, were removed so that the sample size was preserved. De-duplication was performed before imputation, since duplicate records would otherwise bias the group medians used for filling; interpolation was also placed after de-duplication to keep the time axis strictly 1-day spaced.

Informative missingness in survey_score. The 18 missing values (5.00%) in survey_score arise from questionnaire non-response and are not missing at random: respondents with lower scores are less likely to report one. We therefore neither deleted these rows nor imputed them with a mean or median. Deletion would systematically remove low-score respondents and bias the mean, regression coefficients and composite scores upward, while imputation would erase the information carried by non-response itself. Instead, the NaNs were retained and a binary indicator survey_score_missing (1 = non-response) was added, allowing the response behaviour to enter the model explicitly. All 360 observations remain available for analyses that do not use survey_score.

Why rows were not deleted. Row deletion is additionally inappropriate for a daily panel: removing the 18 incomplete rows breaks the strictly 1-day spacing (2-day gaps appear) and shifts regional sample shares by up to 0.73 percentage points, which would distort interpolation, time-series regression and clustering. For the models that require survey_score as an input, we use complete cases within that step only (n = 342) and conduct a sensitivity analysis under two extreme assumptions: (i) non-respondents take the 25th percentile of observed scores, and (ii) they take the regional mean. [fill in: state whether the ranking / coefficients / conclusions remain unchanged] This indicates [fill in: degree of robustness].

⚠️ 最后那两句必须换成你实际跑出来的结果，不能照抄"结论稳健"——这是评审最容易追查的地方，也是学术诚信的底线。

处理摘要表（评审扫一眼就懂，建议放正文或 Appendix）

Variable | Missing rate | Mechanism / cause | Treatment | Rationale
--- | --- | --- | --- | ---
`household_income` | 88.61% | Almost uncollected | Drop column | Preserves n = 360
`pm25` | 4.44% | Missing markers; not recorded | Region-wise median | Respects regional heterogeneity
`temperature` | 1.11% | Equipment outage (consecutive) | Linear interpolation | Keeps 1-day spacing
`traffic_index` | 2.50% | Not recorded | Median | Robust to outliers
`fuel_type` | 2.22% | Blank questionnaire item | Mode | Categorical variable
`survey_score` | 5.00% | Non-response (MNAR) | Keep NaN + binary indicator | Missingness is informative


Assumptions & Limitations 里补一条：We treat non-response in survey_score as informative rather than ignorable; all conclusions involving this variable are conditional on the sensitivity analysis reported above.

中文对照这几段的逻辑：第 1 段讲顺序（规范化 → 去重 → 填补，理由是重复行会拉偏分组中位数、重复日期会打乱插值）；第 2 段讲缺失机制是 MNAR，说明"不删也不填"是主动决策，不是遗漏；第 3 段用实测证据支撑 —— 我这边跑出来删行后间隔从 1 种变成 2 种（最大 2 天）、区域占比 ±0.73 个百分点、pm25 均值 51.51→51.59，这些数字可以直接引用，但记得在你自己的最终数据上复核一遍。

翻译
所有非标准缺失标记（NA、缺测、未填、9999 等）统一转成 NaN，去重前先去掉文本字段首尾空格。删掉 15 条完全重复行，并按确定性规则处理了 6 组 (date, region) 主键冲突：保留有效字段更多、且与整体中位数偏离最小的那条。最终是 360 条日度数据（4 个区域 × 90 个连续日）。缺失率 = 缺失值个数 ÷ 样本总数 ×100%（n=360）。household_income 缺失 88.61%，直接删列——只删列不删行，样本量不变。去重放在填补之前，因为重复记录会拉偏用于填补的分组中位数；插值也放在去重之后，保证时间轴严格保持 1 天等间隔。

survey_score 的"有信息缺失"​

这 18 个缺失（5.00%）来自问卷拒答，而且不是随机漏填：分数低的受访者更不愿填。所以既不删行、也不用均值或中位数填补。删行等于把低分样本系统性剔掉，均值、回归系数、综合得分会偏高；填补则会抹掉"拒答"本身携带的信息。做法是保留 NaN，另加一列 0/1 指示变量 survey_score_missing（1 = 拒答），让"有没有作答"也能进模型。不涉及这个变量的分析，仍可用全部 360 条数据。

为什么不能删行

日度面板本来就不宜删行：删掉 18 行会破坏 1 天等间隔（出现 2 天跳空），各区域样本占比最多偏移 0.73 个百分点，插值、时间序列回归、聚类都会受影响。只有确实要拿 survey_score 当输入的模型，才在那一步内部用完整样本（n=342），并做敏感性分析：假设①拒答者取观测分数的 25 分位数、②取所在区域均值，两种极端情况各跑一遍。〔填：排序/系数/结论是否不变〕，说明〔填：结论的稳健程度〕。

| 变量 | 缺失率 | 成因 | 处理方式 | 理由 |
| --- | --- | --- | --- | --- |
| `household_income` | 88.61% | 几乎未采集 | 删列 | 保住 n = 360 |
| `pm25` | 4.44% | 特殊符号、未记录 | 按区域中位数填补 | 尊重区域差异 |
| `temperature` | 1.11% | 设备故障连续断测 | 线性插值 | 保持 1 天等间隔 |
| `traffic_index` | 2.50% | 未记录 | 中位数填补 | 抗异常值 |
| `fuel_type` | 2.22% | 问卷空填 | 众数填补 | 分类变量 |
| `survey_score` | 5.00% | 问卷拒答，非随机缺失 | 保留 NaN + 0/1 指示变量 | 缺失本身携带信息 |

假设与局限：我们把 survey_score 的拒答当作"带信息的缺失"而非可忽略的缺失；凡涉及该变量的结论，都以上述敏感性分析为前提。

- 类型错误时常用
  - astype()
  - to_numeric()转为数值型
  - to_datatime()转为时间序列
- 建模前检查常用
  - df.info()信息
  - df.describe()描述性统计
  - df.isna().sum()计算缺失值总数
  - df.duplicated().sum()重复行总数

- 异常值：数据错误或真实存在极端情况
  - 1、录入错误：多输数字、单位填错、符号写反
  - 2、真实极端样本：超大城市、极端天气、特殊年份
  - 3、模型敏感点：数值本身没错，但是显著影响均值、回归线或聚类结果，一般用到的少
- 异常值不是一发现就删除，先判断为什么异常，是录入还是真实值异常（核心）
- 录入异常：通勤1200分钟，GDP多个0，通常需要修正和删除
- 真实极端样本：虽然极端，但可能是真实现象，需要解释，例如上海重庆交通复杂，通勤时间长正常

- 用3σ原则判断数据是否异常，所有数据应在μ±3σ之间
  - 1、计算μ均值σ标准差，注意标准差根号里面分母为n-1
  - 2、得到区域
  - 3、判断数据点是否位于±3σ区域

In [ ]:
# -*- coding: utf-8 -*-
#3σ 异常值检测
from pathlib import Path
import pandas as pd
OUT_DIR = Path(r"d:\git项目\输出数据")
NUM = ["pm25", "temperature", "traffic_index", "survey_score"]
GROUP = "region"

def load_any(folder, stem):
    for ext in (".csv", ".xlsx"):
        p = folder / f"{stem}{ext}"
        if p.exists():
            return pd.read_excel(p) if ext == ".xlsx" else pd.read_csv(p)
    raise FileNotFoundError(stem)

def flag_3sigma(s, iters=3):          # 迭代 3σ，避免异常值拉大 σ 造成掩蔽
    m = pd.Series(False, index=s.index)
    for _ in range(iters):
        cur = s[s.notna() & ~m]
        mu, sd = cur.mean(), cur.std(ddof=1)
        if pd.isna(sd) or sd == 0: break
        new = s.notna() & ((s - mu).abs() > 3 * sd)
        if new.equals(m): break
        m = new
    return m

df = load_any(OUT_DIR, "final_cleaned")
df = df.drop(columns=[c for c in df.columns if c.startswith("flag_") or c.endswith("_out")])
for c in NUM:
    df[c + "_out"] = df.groupby(GROUP)[c].transform(lambda s: flag_3sigma(s).astype(int))
df["flag_3sigma"] = df[[c + "_out" for c in NUM]].max(axis=1)
print("3σ 各列:", {c: int(df[c + "_out"].sum()) for c in NUM}, "| 异常行:", int(df.flag_3sigma.sum()))
df.to_csv(OUT_DIR / "result_3sigma.csv", index=False, encoding="utf-8-sig")
df.to_excel(OUT_DIR / "result_3sigma.xlsx", index=False, sheet_name="3sigma")


- IQR方法
- 箱线图把Q1,中位数，Q3上下界和异常点全部展示出来
- IQR=Q_3-Q_1
- Q_1为第一四分位数，Q_3为第三四分位数
- IQR表示中间50%数据的跨度
- 认为所有数据应该属于[Q_1-1.5IQR,Q_3+1.5IQR]
- 在区间之外视为异常值
- 由于IQR依赖四分位数，而不是均值标准差，因此不容易被极端值本身拉偏
- 数据比较少常用IQR

In [ ]:
#IQR 异常值检测
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd
OUT_DIR = Path(r"d:\git项目\输出数据")
NUM = ["pm25", "temperature", "traffic_index", "survey_score"]
GROUP = "region"

def load_any(folder, stem):
    for ext in (".csv", ".xlsx"):
        p = folder / f"{stem}{ext}"
        if p.exists():
            return pd.read_excel(p) if ext == ".xlsx" else pd.read_csv(p)
    raise FileNotFoundError(stem)

def flag_iqr(s):                       # 1.5×IQR，按 region 分组
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return s.notna() & ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr))

df = load_any(OUT_DIR, "final_cleaned")
df = df.drop(columns=[c for c in df.columns if c.startswith("flag_") or c.endswith("_out")])
for c in NUM:
    df[c + "_out"] = df.groupby(GROUP)[c].transform(lambda s: flag_iqr(s).astype(int))
df["flag_iqr"] = df[[c + "_out" for c in NUM]].max(axis=1)
print("IQR 各列:", {c: int(df[c + "_out"].sum()) for c in NUM}, "| 异常行:", int(df.flag_iqr.sum()))
df.to_csv(OUT_DIR / "result_iqr.csv", index=False, encoding="utf-8-sig")
df.to_excel(OUT_DIR / "result_iqr.xlsx", index=False, sheet_name="IQR")


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

OUT_DIR = Path(r"d:\git项目\输出数据")
NUM = ["pm25", "temperature", "traffic_index", "survey_score"]
COLORS = ["#8ecae6", "#a7d7a5", "#ffd28a", "#f4a6a6"]

def load_any(folder, stem):
    for ext in (".csv", ".xlsx"):
        p = folder / f"{stem}{ext}"
        if p.exists():
            return pd.read_excel(p) if ext == ".xlsx" else pd.read_csv(p)
    raise FileNotFoundError(stem)

def draw(method):
    stem = "result_iqr" if method == "IQR" else "result_3sigma"
    df = load_any(OUT_DIR, stem)
    regions = sorted(df["region"].dropna().unique())
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), dpi=120)
    for ax, c in zip(axes.flat, NUM):
        groups = [df.loc[df["region"] == r, c].dropna().values for r in regions]
        bp = ax.boxplot(groups, positions=range(1, len(regions) + 1), widths=0.5,
                        patch_artist=True, showfliers=False, showmeans=True,
                        medianprops={"color": "#c0392b", "linewidth": 2},
                        meanprops={"marker": "D", "markerfacecolor": "#2c3e50",
                                   "markeredgecolor": "#2c3e50", "markersize": 5})
        for box, color in zip(bp["boxes"], COLORS):
            box.set(facecolor=color, edgecolor="#333333", alpha=0.9)
        for key in ("whiskers", "caps"):
            for line in bp[key]:
                line.set(color="#555555", linewidth=1.2)
        for i, r in enumerate(regions, start=1):
            sub = df[(df["region"] == r) & (df[c + "_out"] == 1)]
            if len(sub):
                ax.scatter([i] * len(sub), sub[c], s=45, facecolors="none",
                           edgecolors="#e67e22", linewidths=1.5, zorder=3)
        ax.set_xticks(range(1, len(regions) + 1), regions)
        ax.set_title(c, fontsize=11)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
    fig.suptitle(f"按区域分组箱线图｜橙圈 = {method} 标记的异常（箱体 IQR、红线中位数、菱形均值）", fontsize=13)
    fig.tight_layout()
    out = OUT_DIR / f"boxplot_{method}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("已输出：", out)

draw("IQR")#绘制IQR异常值检测箱线图
draw("3sigma")#绘制3σ异常值检测箱线图


- 先看中位数，再看箱体宽度，最后看是否存在离群点
- 箱体越宽说明波动越大；异常点需要结合业务背景判断
- 用途：发现异常值，比较不同组数据分布差异

- 异常值处理方式：
  - 1、删除：明确录入错误，删除后不影响样本代表性
  - 2、替换：有可靠修正值，或用分组统计量填补
  - 3、缩尾：把超过上下界值压缩到边界，降低极端值影响
  - 例xi*=min(xi,Q3+1.5IQR)表示压缩超过的异常值为上界
  - 4、保留并说明：异常值为真实样本，可保留并在论文中说明